<a href="https://colab.research.google.com/github/mani-m-mp/my-archive/blob/main/Copy_of_Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

In [ ]:
!nvidia-smi

Wed Oct 21 09:14:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.5 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_id = "Qwen/Qwen3-8B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

print("Model loaded successfully")

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded successfully


# New Section

In [ ]:
def ask_model(question):
    messages = [
        {
            "role": "system",
            "content": "تو دستیار دانشگاه هستی. فارسی دقیق بنویس و اطلاعات جعلی تولید نکن."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


print(ask_model("تفاوت RAG و آموزش مجدد مدل زبانی چیست؟"))

تفاوت **RAG (Retrieval-Augmented Generation)** و **آموزش مجدد مدل زبانی (Fine-tuning)** در حوزه هوش مصنوعی و پردازش زبان طبیعی به اینجا می‌رسد:

---

### 1. **آموزش مجدد مدل زبانی (Fine-tuning)**

- **تعریف**: آموزش مجدد یعنی یک مدل زبانی پیش‌آموزش داده شده (مانند BERT، GPT، T5 و غیره) را با داده‌های خاصی که می‌خواهیم مدل برای آن کار کند، مجدد آموزش داد. به عبارت دیگر، مدل را به گونه‌ای تغییر می‌دهیم که بیشتر به نیازهای مسئله خاص (مانند پاسخ به سوال، تولید متن، ترجمه و غیره) پاسخ دهد.

- **مزایا**:
  - مدل به صورت مستقیم با داده‌های مسئله آموزش داده می‌شود.
  - عملکرد مدل در مسئله مورد نظر بهبود می‌یابد.
  - مدل می‌تواند به صورت مستقل از داده‌های خارجی، پاسخ دهد.

- **معایب**:
  - نیاز به داده‌های معتبر و کافی برای آموزش.
  - مدل محدود به داده‌های آموزشی است و اطلاعات خارج از آن را نمی‌تواند به صورت مستقیم استفاده کند.
  - مدل ممکن است اطلاعات نادرست یا غلط را تولید کند، اگر داده‌های آموزشی دارای اشتباه باشد.

---

### 2. **RAG


In [ ]:
import time

# Measure how long one answer takes and how fast the model generates
start = time.time()
answer = ask_model("آیین‌نامه آموزشی چه چیزی را تعیین می‌کند؟")
elapsed = time.time() - start

n_tokens = len(tokenizer.encode(answer))
print(answer)
print(f"\nTime: {elapsed:.1f}s | tokens: {n_tokens} | speed: {n_tokens / elapsed:.1f} tokens/s")

آیین‌نامه آموزشی چه چیزی را تعیین می‌کند؟

آیین‌نامه آموزشی، یک دستورالعمل رسمی و مبنی بر اصول آموزشی است که به منظور تنظیم و سازماندهی فرآیند آموزش و یادگیری در موسسات آموزشی مانند دانشگاه‌ها، مدارس و مراکز آموزشی، تدوین می‌شود. این آیین‌نامه به چندین چیز اصلی می‌پردازد که عبارتند از:

1. **اهداف آموزشی**: تعیین اهداف و محتوای آموزشی که باید در یک دوره یا یک رشته تحصیلی پوشش داده شود.

2. **ساختار و محتوای درسی**: تعیین محتوای درسی، تعداد واحد، ترتیب درس‌ها و زمان‌بندی آموزشی.

3. **روش‌های تدریس و یادگیری**: روش‌های آموزشی مانند تدریس حضوری، آنلاین، گروهی، فردی و سایر روش‌های یادگیری را تعیین می‌کند.

4. **ارزیابی و چارچوب ارزشیابی**: تعیین روش‌های ارزیابی دانشجویان، مانند امتحانات، پروژه‌ها، گزارش‌ها، حضور و غیاب، و معیارهای ارزیابی.

5. **حقوق و مسئولیت‌های دانشجو و معلم**: تعیین می‌کند که هر کدام از طرفین (دانشجو و معلم)

Time: 63.7s | tokens: 511 | speed: 8.0 tokens/s
